# Steam Data Collection

Collects the reviews, discussion posts and comments of the Steam pages listed in `steam_links.txt`. Steam has no search over reviews and comments, so the pages are chosen by hand, as for Instagram and TikTok.

No Steam account or login is needed: the data is read from public Steam Community pages, the comment endpoint those pages use, and Steam's public review API.

Supported links:
1. A profile's review list (`https://steamcommunity.com/id/<name>/recommended/`): every review on all its pages, each with its comments.
2. A single review (`.../recommended/<app id>/`): the review and its comments.
3. A profile or its comments page (`.../allcomments`): all profile comments.
4. A group (`https://steamcommunity.com/groups/<name>`): all group comments.
5. A discussion forum (`https://steamcommunity.com/app/<app id>/discussions/`): the newest topics, each with its replies.
6. A discussion topic (`.../discussions/<forum>/<topic>/`): the first post and all replies.
7. A store page (`https://store.steampowered.com/app/<app id>/`) or a community review page (`https://steamcommunity.com/app/<app id>/reviews/`, which only opens with a login): store reviews of the game from the past year, as many as Steam's review API returns. For some games, such as Dota 2, the API returns only a few dozen.

Event comments, group announcements and other pages with a Steam comment thread are collected like a profile. A page with no review, discussion post or comment thread is reported as a failed link.

Each link is saved as soon as it is collected: its new texts are added to `SteamComments.csv` and the link is added to `SteamComments.progress.json`. Links in the progress file are skipped on later runs, so only links that are new or failed before are collected, and a text already in `SteamComments.csv` is never added again.

Page and language parameters in a link (`?p=2`, `?ctp=3`, `?l=malay`) are ignored, since all pages are collected. Steam has no Persian review language, and most texts on large forums and store pages are in other languages, so by default only Persian texts are kept. A text counts as Persian when it is in Arabic script but has none of the letters only Arabic uses (ة and the Arabic forms of ي and ك), so Persian typed with an Arabic keyboard layout and short Arabic texts can be misjudged. Private profiles are reported as failed links.

In [ ]:
import itertools
import json
import os
import re
import time
from urllib.parse import urlsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Configuration

In [ ]:
LINKS_FILE = "steam_links.txt"
OUTPUT_CSV = "SteamComments.csv"
PROGRESS_FILE = "SteamComments.progress.json"  # links in it are skipped
KEEP_ONLY_PERSIAN = True
MAX_COMMENTS_PER_THREAD = 2000
MAX_REVIEWS_PER_PROFILE = 200
MAX_TOPICS_PER_FORUM = 30
MAX_REVIEWS_PER_GAME = 2000
REVIEW_LANGUAGES = ['english', 'arabic']  # Steam has no Persian review language
DELAY_BETWEEN_REQUESTS = 3  # seconds
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

## Read links

In [ ]:
def read_links(path):
    with open(path, 'r', encoding='utf-8') as links_file:
        lines = [line.strip() for line in links_file]
    return [line for line in lines if line and not line.startswith('#')]

def normalize_link(link):
    # ?p=, ?ctp= and ?l= are dropped since every page is collected
    parts = urlsplit(link)
    return 'https://' + parts.netloc + parts.path.rstrip('/') + '/'

def read_done_links(path):
    if not os.path.exists(path):
        return []
    with open(path, 'r', encoding='utf-8') as progress_file:
        return json.load(progress_file)

def save_done_links(path, done_links):
    with open(path, 'w', encoding='utf-8') as progress_file:
        json.dump(done_links, progress_file, ensure_ascii=False, indent=1)

links = list(dict.fromkeys(normalize_link(link) for link in read_links(LINKS_FILE)))
done_links = read_done_links(PROGRESS_FILE)
links_to_collect = [link for link in links if link not in done_links]
print(len(links), 'links,', len(links) - len(links_to_collect), 'already collected,', len(links_to_collect), 'to collect')

## Request Steam pages

In [ ]:
def make_session():
    # Steam can keep answering 429 for several minutes
    retry = Retry(total=8, backoff_factor=10, backoff_max=300, status_forcelist=[429, 500, 502, 503, 504])
    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=retry))
    session.headers.update({'User-Agent': USER_AGENT})
    return session

def get(session, url, params=None):
    time.sleep(DELAY_BETWEEN_REQUESTS)
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response

def get_page(session, url, params=None):
    # English pages give one date format
    html = get(session, url, {'l': 'english', **(params or {})}).text
    soup = BeautifulSoup(html, 'html.parser')
    error = soup.select_one('#message h3')
    if error is not None:
        raise ValueError('Steam error: ' + error.get_text(strip=True))
    if soup.select_one('.profile_private_info') is not None:
        raise ValueError('The profile is private')
    return html, soup

## Get reviews, discussion posts and comments

In [ ]:
THREAD_PATTERN = re.compile(r'InitializeCommentThread\(\s*"\w+",\s*"[^"]*",\s*')
THREAD_URL_PATTERN = re.compile(r"\s*,\s*'([^']+)'")
COMMENTS_PER_REQUEST = 50  # Steam's maximum

def find_comment_threads(html):
    # e.g. InitializeCommentThread( "Profile", "Profile_7656...", {"owner":"7656...",...}, 'https://steamcommunity.com/comment/Profile/', 40 )
    threads = []
    for match in THREAD_PATTERN.finditer(html):
        settings, end = json.JSONDecoder().raw_decode(html, match.end())
        url_match = THREAD_URL_PATTERN.match(html, end)
        if url_match is None:
            raise ValueError('Unknown comment thread format: ' + html[match.start():end + 100])
        render_url = url_match.group(1) + 'render/' + settings['owner'] + '/' + settings['feature'] + '/'
        params = {'feature2': settings['feature2'], 'extended_data': settings.get('extended_data', '')}
        threads.append((render_url, params))
    return threads

def iter_thread_comments(session, render_url, params):
    start = 0
    while True:
        page = get(session, render_url, {**params, 'start': start, 'count': COMMENTS_PER_REQUEST}).json()
        if not page.get('success'):
            raise ValueError('Steam did not return the comments of ' + render_url + ': ' + str(page.get('error')))
        comments = BeautifulSoup(page['comments_html'], 'html.parser').select('.commentthread_comment')
        yield from comments
        start += len(comments)
        if not comments or start >= page['total_count']:
            return

In [ ]:
REVIEW_ID_PATTERN = re.compile(r"UserReviewVoteUp\(\s*\d+,\s*'[^']*',\s*'(\d+)'")
REVIEW_DATE_PATTERN = re.compile(r'Posted: (\d{1,2} \w+)(?:, (\d{4}))? @ (\d{1,2}:\d{2}[ap]m)')

def element_text(element):
    # Quoted earlier posts are removed so they are not collected twice
    for quote in element.select('blockquote'):
        quote.decompose()
    for line_break in element.select('br'):
        line_break.replace_with('\n')
    return element.get_text().strip()

def timestamp_of(element):
    return pd.to_datetime(int(element.select_one('[data-timestamp]')['data-timestamp']), unit='s')

def parse_review_date(text):
    # e.g. 'Posted: 11 Dec, 2022 @ 12:26pm'; the year is left out for the current year
    match = REVIEW_DATE_PATTERN.search(text)
    if match is None:
        raise ValueError('Unknown review date format: ' + text)
    day_month, year, clock = match.groups()
    return pd.to_datetime(f'{day_month} {year or pd.Timestamp.now().year} {clock}')

def text_row(link, page_url, source, text_id, username, created_at, text):
    return {'link': link, 'page_url': page_url, 'source': source, 'id': text_id, 'username': username,
            'created_at': created_at, 'text': text}

def comment_row(link, page_url, comment):
    return text_row(link, page_url, 'comment', comment['id'].removeprefix('comment_'),
                    comment.select_one('.commentthread_author_link').get_text(strip=True), timestamp_of(comment),
                    element_text(comment.select_one('.commentthread_comment_text')))

def review_row(link, page_url, html, soup):
    review = soup.select_one('#ReviewText')
    if review is None:
        return None
    review_id = REVIEW_ID_PATTERN.search(html)
    if review_id is None:
        raise ValueError('No review id in ' + page_url)
    return text_row(link, page_url, 'review', review_id.group(1),
                    soup.select_one('.profile_small_header_name').get_text(strip=True),
                    parse_review_date(soup.select_one('.recommendation_date').get_text(' ', strip=True)),
                    element_text(review))

def topic_row(link, page_url, soup):
    topic = soup.select_one('.forum_op')
    if topic is None:
        return None
    text = element_text(topic.select_one('.topic')) + '\n' + element_text(topic.select_one('.content'))
    author = topic.select_one('.forum_op_author')  # missing on event posts
    username = author.get_text(strip=True) if author is not None else None
    return text_row(link, page_url, 'topic', topic['id'].removeprefix('forum_op_'), username, timestamp_of(topic), text)

def get_page_texts(session, link, page_url):
    html, soup = get_page(session, page_url)
    rows = [row for row in (review_row(link, page_url, html, soup), topic_row(link, page_url, soup)) if row is not None]
    threads = find_comment_threads(html)
    if not rows and not threads:
        raise ValueError('No review, discussion post or comment thread on ' + page_url)
    for render_url, params in threads:
        comments = itertools.islice(iter_thread_comments(session, render_url, params), MAX_COMMENTS_PER_THREAD)
        rows.extend(comment_row(link, page_url, comment) for comment in comments)
    return rows

In [ ]:
GAME_REVIEWS_URL = 'https://api.steampowered.com/IUserReviewsService/GetAppReviews/v1/'
# Community review pages need a login, so their reviews are read from the API like store pages
GAME_PATTERN = re.compile(r'store\.steampowered\.com/app/(\d+)/|steamcommunity\.com/app/(\d+)/(?:positive|negative)?reviews/$')
FORUM_PATTERN = re.compile(r'steamcommunity\.com/(?:app|groups)/[^/]+/discussions/(?:\d+/)?$')
REVIEW_LIST_PATTERN = re.compile(r'steamcommunity\.com/(?:id|profiles)/[^/]+/recommended/$')

def iter_list_links(session, list_url, page_param, link_selector):
    # A page past the last one is empty
    for page in itertools.count(1):
        _, soup = get_page(session, list_url, {page_param: page})
        item_links = [anchor['href'] for anchor in soup.select(link_selector)]
        if not item_links:
            return
        yield from item_links

def iter_game_reviews(session, app_id):
    # Without day_range the API only covers the last 30 to 90 days
    params = {'appid': app_id, 'num_per_page': 100, 'cursor': '*', 'filter': 'recent', 'day_range': 365}
    params.update({f'languages[{i}]': language for i, language in enumerate(REVIEW_LANGUAGES)})
    while True:
        page = get(session, GAME_REVIEWS_URL, params).json()['response']
        reviews = page.get('reviews', [])
        yield from reviews
        if not reviews or page.get('cursor', params['cursor']) == params['cursor']:
            return
        params['cursor'] = page['cursor']

def game_review_row(link, app_id, review):
    steam_id = review['author']['steamid']
    page_url = f'https://steamcommunity.com/profiles/{steam_id}/recommended/{app_id}/'
    return text_row(link, page_url, 'review', review['recommendationid'], steam_id,
                    pd.to_datetime(review['timestamp_created'], unit='s'), review['review'])

def get_link_texts(session, link):
    game = GAME_PATTERN.search(link)
    if game:
        app_id = game.group(1) or game.group(2)
        reviews = itertools.islice(iter_game_reviews(session, app_id), MAX_REVIEWS_PER_GAME)
        return [game_review_row(link, app_id, review) for review in reviews]
    if FORUM_PATTERN.search(link):
        page_urls = itertools.islice(iter_list_links(session, link, 'fp', 'a.forum_topic_overlay'), MAX_TOPICS_PER_FORUM)
    elif REVIEW_LIST_PATTERN.search(link):
        page_urls = itertools.islice(iter_list_links(session, link, 'p', '.review_box .title a'), MAX_REVIEWS_PER_PROFILE)
    elif 'steamcommunity.com/' in link:
        page_urls = [link]
    else:
        raise ValueError('Not a supported Steam link: ' + link)
    rows = []
    for page_url in page_urls:
        rows.extend(get_page_texts(session, link, page_url))
    return rows

## Save texts

In [ ]:
TEXT_COLUMNS = ['link', 'page_url', 'source', 'id', 'username', 'created_at', 'text']
ARABIC_SCRIPT_LETTERS = re.compile('[ء-يٮ-ۓ]')
ARABIC_ONLY_LETTERS = re.compile('[ةيك]')  # Persian writes ی and ک instead

def is_persian(text):
    return bool(ARABIC_SCRIPT_LETTERS.search(text)) and not ARABIC_ONLY_LETTERS.search(text)

def clean_text(text):
    # same cleaning as the other notebooks
    text = text.replace('#', '').replace(',', ' ').replace('\r', ' ').replace('\n', ' ')
    return re.sub(r'@[^\s]+', '', text).strip()

def read_saved_texts(path):
    if not os.path.exists(path):
        return pd.DataFrame(columns=TEXT_COLUMNS)
    return pd.read_csv(path, index_col=0, dtype={'id': str}, encoding='utf-8')

def select_new_texts(link_texts, saved_keys):
    # A text is identified by its source and Steam id
    texts = pd.DataFrame(link_texts, columns=TEXT_COLUMNS).drop_duplicates(subset=['source', 'id'])
    texts['text'] = texts['text'].apply(clean_text)
    keep = [text != '' and (is_persian(text) or not KEEP_ONLY_PERSIAN) and (source, text_id) not in saved_keys
            for source, text_id, text in zip(texts['source'], texts['id'], texts['text'])]
    return texts.loc[keep]

def append_texts(path, texts, first_index):
    texts.index = range(first_index, first_index + len(texts))
    texts.to_csv(path, mode='a', header=not os.path.exists(path), index=True, encoding='utf-8')

## Collect the links

In [ ]:
session = make_session()
saved_texts = read_saved_texts(OUTPUT_CSV)
saved_keys = set(zip(saved_texts['source'], saved_texts['id']))
saved_count = len(saved_texts)
failed_links = []
for i, link in enumerate(links_to_collect):
    print(i, '--------------- getting texts of', link, '------------------')
    try:
        link_texts = get_link_texts(session, link)
    except (requests.RequestException, ValueError) as err:
        print('failed:', err)
        failed_links.append(link)
        continue
    new_texts = select_new_texts(link_texts, saved_keys)
    append_texts(OUTPUT_CSV, new_texts, saved_count)
    saved_keys.update(zip(new_texts['source'], new_texts['id']))
    saved_count += len(new_texts)
    done_links.append(link)
    save_done_links(PROGRESS_FILE, done_links)
    print('got', len(link_texts), 'texts,', len(new_texts), 'new ones saved')

print('texts in', OUTPUT_CSV + ':', saved_count, '| failed links:', len(failed_links))

In [ ]:
steam_df = read_saved_texts(OUTPUT_CSV)
steam_df.head(900)